# 02 — Stage 1, Step 5: BM25 baseline on a 100K MS-MARCO subset

Goal: index 100K passages with `bm25s`, run one dev query, eyeball the top-10.

Assumes `01_load_msmarco.ipynb` already populated the Drive cache with the full corpus.

This notebook gives you the **read-from-Drive scaffolding**. The tokenize / index /
retrieve cells are stubs — you fill them in (Step 5 is yours to write).

In [ ]:
# Universal bootstrap — same in every notebook.
import sys, os
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    if not os.path.exists('Information-Retreival'):
        !git clone https://github.com/ZiaadNegm/Information-Retreival.git
    %cd Information-Retreival
    %pip install -q -r requirements.txt
else:
    sys.path.insert(0, os.path.abspath('..'))

from bootstrap import setup_env
setup_env()

In [ ]:
# Load handles from cache (instant — corpus already on disk from 01_load_msmarco).
import ir_datasets

docs_ds = ir_datasets.load('msmarco-passage')
dev_ds  = ir_datasets.load('msmarco-passage/dev/small')

print('docs   :', docs_ds.docs_count())
print('queries:', dev_ds.queries_count())
print('qrels  :', dev_ds.qrels_count())

In [ ]:
# Where to keep persistent artifacts (100K JSONL, BM25 index, run files).
# On Colab → Drive. Locally → repo-relative ./data/.
from pathlib import Path

if IN_COLAB:
    ARTIFACTS = Path('/content/drive/MyDrive/ir-roadmap-cache/artifacts')
else:
    ARTIFACTS = Path(__file__).resolve().parent.parent / 'data' if '__file__' in dir() else Path('../data')

ARTIFACTS.mkdir(parents=True, exist_ok=True)
CORPUS_PATH = ARTIFACTS / 'corpus_100k.jsonl'
INDEX_DIR   = ARTIFACTS / 'bm25_index_100k'

print('artifacts:', ARTIFACTS)
print('corpus   :', CORPUS_PATH, '(exists)' if CORPUS_PATH.exists() else '(will create)')
print('index    :', INDEX_DIR,   '(exists)' if INDEX_DIR.exists()   else '(will create)')

In [ ]:
# Sample 100K passages from the corpus → write JSONL once. Idempotent: skips if file exists.
# Strategy: deterministic first-100K via itertools.islice (fast, reproducible).
# If you want a uniform random sample, swap to random.sample over doc_ids with a fixed seed.
import json, itertools
from tqdm.auto import tqdm

N_SAMPLE = 100_000

if CORPUS_PATH.exists():
    print(f'corpus_100k.jsonl already exists at {CORPUS_PATH} — skipping sample.')
else:
    with open(CORPUS_PATH, 'w') as f:
        for doc in tqdm(itertools.islice(docs_ds.docs_iter(), N_SAMPLE), total=N_SAMPLE):
            f.write(json.dumps({'doc_id': doc.doc_id, 'text': doc.text}) + '\n')
    size_mb = CORPUS_PATH.stat().st_size / 1e6
    print(f'wrote {CORPUS_PATH}  ({size_mb:.1f} MB)')

In [ ]:
# Reload pattern — every later cell uses these two parallel lists.
# IMPORTANT: bm25s returns POSITIONAL indices; keep doc_ids[i] alongside texts[i]
# so you can resolve hits back to MS-MARCO doc IDs.
doc_ids: list[str] = []
texts:   list[str] = []

with open(CORPUS_PATH) as f:
    for line in f:
        rec = json.loads(line)
        doc_ids.append(rec['doc_id'])
        texts.append(rec['text'])

print('loaded:', len(doc_ids), 'passages')
print('first :', doc_ids[0], '|', texts[0][:120])

---
## Your work below — Step 5 deliverable

Stubs only. Fill in using the `bm25s` API:
- `bm25s.tokenize(list_of_texts)` → `Tokenized` object
- `retriever = bm25s.BM25(k1=0.9, b=0.4)` then `retriever.index(tokenized_corpus)`
- `results, scores = retriever.retrieve(query_tokens, k=10)`
- `retriever.save(INDEX_DIR)` so Step 7 can reuse the index without re-tokenising

In [ ]:
# Step 5a — Tokenize the 100K corpus (1-3 min on Colab CPU).
import bm25s

# tokenized_corpus = bm25s.tokenize(texts)
# print(type(tokenized_corpus))


In [ ]:
# Step 5b — Build the BM25 index, save it for reuse.

# retriever = bm25s.BM25(k1=0.9, b=0.4)
# retriever.index(tokenized_corpus)
# retriever.save(str(INDEX_DIR))


In [ ]:
# Step 5c — Pick one dev query, retrieve top-10, print with eyeball judgments.

# q = next(iter(dev_ds.queries_iter()))
# print('query:', q.query_id, '|', q.text)
#
# query_tokens = bm25s.tokenize([q.text])
# results, scores = retriever.retrieve(query_tokens, k=10)
#
# for rank, (idx, score) in enumerate(zip(results[0], scores[0]), start=1):
#     print(f'{rank:2d}  doc_id={doc_ids[idx]:>10}  score={score:.3f}')
#     print(f'    {texts[idx][:200]}')
#     print()
